In [50]:
import os
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from pprint import pprint

from utils.analyses.output_data_preprocess import *
from utils.analyses.descriptives import *
from utils.analyses.ref_letters_analysis import *
from utils.analyses.psychometrics import *

In [51]:
# set all needed directories
current_dir = os.getcwd()
output_data_dir = os.path.join(current_dir, "output_data")
results_dir = os.path.join("C:/Users/jana/Documents/Repos/measuring-sexism-in-LLMs", "results")

In [52]:
def preprocess_test_data(task):
    df = load_and_concat_jsons(base_dir=output_data_dir, subfolder=task, file_suffix=task)

    # count NaN answers for every model
    nan_counts = df.groupby('model_name')['answer_reversed'].apply(lambda x: x.isna().sum()).reset_index()
    nan_counts.columns = ['model_name', 'nan_count']
    # save in json
    nan_counts.to_json(os.path.join(results_dir,task,"nan_count_per_model.json"), orient="columns", indent=2)

    # for SR2K: transform answers to item 3 from scale 1-3 to scale 1-4
    if task == "SR2K":
        item_3 = df["item_id"] == 3
        df.loc[item_3, "answer_reversed"] = 1.5 * df.loc[item_3, "answer_reversed"] - 0.5
        df["answer_reversed"] = 4 - df["answer_reversed"] + 1
    # calculate mean and sd over the different seeds for each item
    #df_agg = df.groupby(["model_name", "seeds", "item_id", "subscale", "reversed"], as_index=False).agg(avg_answer_reversed=('answer_reversed', 'mean'))

    avg_scores = (
        df.groupby(["model_name", "seed"])["answer_reversed"]
        .mean()
        .reset_index(name="total")
    )

    return avg_scores

In [53]:
def r_by_seed(merged:pd.DataFrame, construct:str):
    results = []

    for seed, group in merged.groupby("seed"):
        # compute the correlation for this seed only
        r, lower, upper = spearman_rank_corr(
            group[f"{construct}_score"], 
            group["total"], 
            alternative="greater"
        )
        
        results.append({
            "seed": seed,
            "spearman_r": r,
            "lower_CI": lower,
            "upper_CI": upper
        })

    pprint(results)

    return results

In [54]:
def r_avg_std(results:dict):
    r_values = [item["spearman_r"].statistic for item in results]

    # compute the average correlation across seeds
    avg_r = np.mean(r_values)
    std_r = np.std(r_values)

    print("Average Spearman correlation across seeds:", avg_r, "(std:", std_r, ")")

# Sexism

In [55]:
df_ASI = preprocess_test_data("ASI")

c:\Users\jana\Documents\Repos\measuring-sexism-in-LLMs\src\output_data\ASI


In [56]:
df_ref = load_and_concat_jsons(base_dir=output_data_dir, subfolder="ref_letter_generation", file_suffix="")

df_ref_wide = df_ref.groupby(["model_name", "seed"]).apply(
    analyze_ref_letters,
    include_groups = False
).reset_index()

# get all columns containing OR values
OR_columns = [col for col in df_ref_wide.columns if "OR" in col]

# calculate overall sexism score for each context by averaging over OR values
df_ref_wide["sexism_score"] = df_ref_wide[OR_columns].mean(axis=1)

c:\Users\jana\Documents\Repos\measuring-sexism-in-LLMs\src\output_data\ref_letter_generation


In [57]:
merged_sexism = pd.merge(left=df_ASI, right=df_ref_wide[["model_name","seed", "sexism_score"]], how="left", on=["model_name", "seed"])
merged_sexism = merged_sexism[merged_sexism.model_name != "Llama-3.1-Centaur-70B"]

In [58]:
results_ASI = r_by_seed(merged_sexism, construct="sexism")

[{'lower_CI': -0.8323180070127484,
  'seed': 1,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.4947274449181537), pvalue=np.float64(0.9489909769623467)),
  'upper_CI': 0.11057871085962032},
 {'lower_CI': -0.74692338302511,
  'seed': 2,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.30282441053771947), pvalue=np.float64(0.8306550461098552)),
  'upper_CI': 0.3281084990608096},
 {'lower_CI': -0.9038470872439365,
  'seed': 3,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.6855508879580129), pvalue=np.float64(0.9930699799863203)),
  'upper_CI': -0.18405764992376142},
 {'lower_CI': -0.8186128293632717,
  'seed': 4,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.46153846153846156), pvalue=np.float64(0.9345259652464136)),
  'upper_CI': 0.15286132640746286},
 {'lower_CI': -0.5878354799849181,
  'seed': 5,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.021015794050150385), pvalue=np.float64(0.525844055751374)),
  'upper_CI': 0.559643831

In [60]:
r_avg_std(results_ASI)

Average Spearman correlation across seeds: -0.3931313998004996 (std: 0.2223222057672733 )


# Racism

In [61]:
df_SR2K = preprocess_test_data("SR2K")

c:\Users\jana\Documents\Repos\measuring-sexism-in-LLMs\src\output_data\SR2K


In [62]:
df_hr = pd.read_csv(os.path.join(results_dir, "ecological", "housing_per_model_seed.csv"))
df_hr = df_hr.rename(columns={"mean_difference":"racism_score"})

In [63]:
merged_racism = pd.merge(left=df_SR2K, right=df_hr[["model_name","seed", "racism_score"]], how="left", on=["model_name", "seed"])
merged_racism = merged_racism[merged_racism.model_name != "Llama-3.1-Centaur-70B"]

In [64]:
results_SR2K = r_by_seed(merged_racism, construct="racism")

[{'lower_CI': -0.9038470872439365,
  'seed': 1,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.6855508879580129), pvalue=np.float64(0.9930699799863203)),
  'upper_CI': -0.18405764992376142},
 {'lower_CI': -0.9401754355032899,
  'seed': 2,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.7954962950610549), pvalue=np.float64(0.9990122070162148)),
  'upper_CI': -0.407735355279729},
 {'lower_CI': -0.9259567854021183,
  'seed': 3,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.7512991913334103), pvalue=np.float64(0.9975757675221594)),
  'upper_CI': -0.31185411707699884},
 {'lower_CI': -0.9491664689812657,
  'seed': 4,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.8242616304086429), pvalue=np.float64(0.9995125596178657)),
  'upper_CI': -0.47509778708300177},
 {'lower_CI': -0.8837026073937396,
  'seed': 5,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.6285942248849762), pvalue=np.float64(0.9857126959433528)),
  'upper_CI': -0.0855457

In [66]:
r_avg_std(results_SR2K)

Average Spearman correlation across seeds: -0.7370404459292195 (std: 0.07160962306710762 )


# Morality

In [67]:
df_MFQ = preprocess_test_data("MFQ")

c:\Users\jana\Documents\Repos\measuring-sexism-in-LLMs\src\output_data\MFQ


In [68]:
df_advice = load_and_concat_jsons(base_dir=output_data_dir, subfolder="advice", file_suffix="")
df_advice = df_advice[df_advice["model_name"] != "Llama-3.1-Centaur-70B"]

#print(df_advice['pro_value'].value_counts())

condition_match = (
    (df_advice['pro_value'] == True) & (df_advice['judge_action_taken'] == 'yes')
) | (
    (df_advice['pro_value'] == False) & (df_advice['judge_action_taken'] == 'no')
)
df_advice['match'] = np.where(condition_match, 1,0)

df_advice_agg = (
        df_advice.groupby(["model_name", "seed", "subscale"])["match"]
        .mean()
        .reset_index(name="subscale_score")
    )

c:\Users\jana\Documents\Repos\measuring-sexism-in-LLMs\src\output_data\advice


In [72]:
subscales = ["authority", "care", "fairness", "ingroup", "purity"]

for sub in subscales:
    print("-----------------------------------------")
    print(sub)
    df_advice_agg_sub = df_advice_agg.loc[df_advice_agg["subscale"] == sub]

    merged_sub = pd.merge(left=df_SR2K, right=df_advice_agg_sub[["model_name","seed", "subscale_score"]], how="left", on=["model_name", "seed"])
    merged_sub = merged_sub[merged_sub.model_name != "Llama-3.1-Centaur-70B"]

    results_sub = r_by_seed(merged_sub, construct="subscale")

    r_avg_std(results_sub)
    

-----------------------------------------
authority
[{'lower_CI': -0.767158645632871,
  'seed': 1,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.34525994422096135), pvalue=np.float64(0.8641515667334254)),
  'upper_CI': 0.2851520349535243},
 {'lower_CI': -0.7310363439656686,
  'seed': 2,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.2706974898386595), pvalue=np.float64(0.8026138615100362)),
  'upper_CI': 0.3589821802613624},
 {'lower_CI': -0.7425667246143453,
  'seed': 3,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.2939127377167585), pvalue=np.float64(0.8231032072689543)),
  'upper_CI': 0.3368097908941224},
 {'lower_CI': -0.5822774329843893,
  'seed': 4,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.012567507241579293), pvalue=np.float64(0.5154606688464847)),
  'upper_CI': 0.5654203192728263},
 {'lower_CI': -0.8186832623839151,
  'seed': 5,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.4617065072933272), pvalue=np.float6

# Convergent